In [3]:
import numpy as np
import time
import json
import os
from datetime import datetime
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

path = "../../../data/binary/processed/mnist_01_no_pca"

In [4]:
seeds = [42, 100, 20]

results_path = "../../../results/classical_logreg_no_pca_results.json"

if os.path.exists(results_path):
    with open(results_path, 'r') as f:
        all_results = json.load(f)
    print(f"Loaded existing results with {len(all_results['results'])} entries")
else:
    all_results = {
        "experiment_info": {
            "model_type": "classical_logistic_regression",
            "date": datetime.now().isoformat(),
            "hyperparameter_tuning": "GridSearchCV with C=[0.001, 0.01, 0.1, 1, 10, 100]",
            "cv_folds": 5
        },
        "results": []
    }
    print("Created new results file")


Created new results file


In [5]:
 # Load full training data
X_train_full = np.load(path + "/X_train.npy")
X_test = np.load(path + "/X_test.npy")
y_train_full = np.load(path + "/y_train.npy")
y_test = np.load(path + "/y_test.npy")

print(f"\n{'='*70}")
print(f"Dataset: mnist_01_no_pca")
print(f"Available training samples: {X_train_full.shape[0]}")
print(f"{'='*70}")


for seed in seeds:
    X_train, _, y_train, _ = train_test_split(
    X_train_full, y_train_full,
    train_size=100,
    random_state= seed,  
    stratify=y_train_full)
    print(f"\n{'─'*70}")
    print(f"{'─'*70}")
# Check if this specific experiment exists
    specific_existing = [r for r in all_results["results"] 
                    if r["dataset"] == "mnist_01_no_pca"
                    and r["n_train"] == 100
                    and r.get("seed") == seed]
            
    if specific_existing:
        print(f"  Seed {seed}: Already exists, skipping...")
        continue

# Hyperparameter tuning with GridSearchCV
    param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}
    log_reg_cv = GridSearchCV(
            LogisticRegression(random_state=seed, max_iter=1000), 
            param_grid, 
            cv=5,
            n_jobs=-1  # Use all CPU cores for faster training
        )

    # Train
    start_time = time.time()
    log_reg_cv.fit(X_train, y_train)
    training_time = time.time() - start_time
    
    # Inference
    start_time = time.time()
    y_pred = log_reg_cv.predict(X_test)
    inference_time = time.time() - start_time
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')

    # Store result
    result = {
                "dataset": "mnist_01_no_pca",
                "n_train": 100,
                "n_test": int(X_test.shape[0]),
                "n_features": int(X_train.shape[1]),
                "seed": int(seed),
                "best_C": float(log_reg_cv.best_params_['C']),
                "accuracy": float(accuracy),
                "f1_score": float(f1),
                "training_time_seconds": float(training_time),
                "inference_time_seconds": float(inference_time),
                "timestamp": datetime.now().isoformat()
            }
    all_results["results"].append(result)
    print(f"  Seed {seed:3d}: Acc={accuracy:.4f}, F1={f1:.4f}, "
                  f"Best C={log_reg_cv.best_params_['C']:.3f}, "
                  f"Train={training_time:.3f}s")



# Save results
os.makedirs("../../../results", exist_ok=True)
with open(results_path, 'w') as f:
    json.dump(all_results, indent=2, fp=f)

print(f"\n{'='*70}")
print(f" Results saved to {results_path}")
print(f"Total experiments: {len(all_results['results'])}")
print(f"{'='*70}")


Dataset: mnist_01_no_pca
Available training samples: 4000

──────────────────────────────────────────────────────────────────────
──────────────────────────────────────────────────────────────────────
  Seed  42: Acc=0.9970, F1=0.9970, Best C=0.100, Train=1.633s

──────────────────────────────────────────────────────────────────────
──────────────────────────────────────────────────────────────────────
  Seed 100: Acc=0.9960, F1=0.9960, Best C=0.010, Train=0.039s

──────────────────────────────────────────────────────────────────────
──────────────────────────────────────────────────────────────────────
  Seed  20: Acc=0.9450, F1=0.9446, Best C=0.001, Train=0.036s

 Results saved to ../../../results/classical_logreg_no_pca_results.json
Total experiments: 3
